# exp03 — Pretrain trên VOYA_VSL (161 lớp, chạy được NGAY không cần chờ dataset)

**Mở trên Colab:** https://colab.research.google.com/github/minKasent/signbridge/blob/main/training/experiments/exp03_voya_pretrain.ipynb

VOYA_VSL (MIT, tải tự do) = 1 video từ điển QIPEDC/lớp + 1000 augment. **Chỉ dùng để
PRETRAIN** — model học "ngôn ngữ chung của chuyển động tay" trước, sau này fine-tune
trên dataset thật (VSL400) bằng `--init-from` sẽ hội tụ nhanh và tốt hơn train từ 0.
Đây cũng là một thí nghiệm so sánh cho chương 4: có/không pretrain.

- **Runtime: T4 GPU.** Dữ liệu tải ~54GB nhưng convert xong chỉ giữ .npy (~2GB) —
  dùng `--delete-npz` để không tràn disk Colab (~78GB).
- Thời gian: tải+convert ~1-2h, train ~1h. Convert resumable (file .npz đã convert bỏ qua
  nếu còn cache; mất phiên thì phần .npy trên Drive còn nguyên).

In [ ]:
# Cell 1 — Chuẩn bị
%pip install -q wandb onnxscript
!git clone -q https://github.com/minKasent/signbridge.git /content/signbridge 2>/dev/null || (cd /content/signbridge && git pull -q)
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CHƯA BẬT")
from google.colab import drive; drive.mount("/content/drive")

In [ ]:
# Cell 2 — Tải + convert 161 lớp VOYA về 144 chiều (~1-2h, resumable)
# .npz tải về disk Colab (xóa ngay sau convert); .npy ghi vào Drive (bền qua phiên)
!python /content/signbridge/training/preprocess/voya_to_npy.py \
    --classes 1-161 --max-aug 200 --delete-npz \
    --cache /content/voya_cache \
    --out "/content/drive/MyDrive/datn/voya_npy"

In [ ]:
# Cell 3 — Đăng nhập W&B rồi pretrain (~1h trên T4)
import wandb; wandb.login()
!cd /content/signbridge/training/model && python train.py \
    --data "/content/drive/MyDrive/datn/voya_npy" \
    --out "/content/drive/MyDrive/datn/models/exp03_voya_pretrain" \
    --epochs 30 --batch 128 --workers 2 --no-sign-per-class 2 \
    --wandb --run-name exp03-voya-pretrain

## Dùng kết quả

Checkpoint nằm ở `Drive/datn/models/exp03_voya_pretrain/best.pt`. Khi có dataset thật
(VSL400), fine-tune trong exp02 chỉ cần thêm:

```
--init-from "/content/drive/MyDrive/datn/models/exp03_voya_pretrain/best.pt" --lr 1e-4
```

và chạy thêm một lần KHÔNG có `--init-from` để có cặp số liệu so sánh cho chương 4.

⚠️ Val accuracy của exp03 KHÔNG có ý nghĩa tổng quát hóa (1 người ký/lớp) — đừng đưa
vào báo cáo như kết quả nhận diện; nó chỉ là chỉ báo model đang học được cấu trúc.